# 2-Class Hand X-Ray Validator Training (Real Data Only)

Trains `HandXRayValidator` (ImageNet-pretrained `ResNet-18`, 2 classes: `["valid_pediatric_hand_xray", "other"]`) on **real data only**.

### Key Guarantees
- **Default Positives:** Up to **5,000** real RSNA hand/wrist radiographs (`MAX_POSITIVES = 5000`).
- **Strict Data-Leakage Prevention:**
  - Automatically excludes all **30** `tests/validator/should_accept` filenames and all **200** Stage 1/3 check filenames (**230** held-out RSNA radiographs) from positive training/validation.
  - Automatically excludes all `tests/validator/should_reject` filenames (and SHA-256 hashes, if `/content/should_reject` or `/content/should_reject.zip` is uploaded) from negative training/validation.
- **Detailed Logging:** Logs raw counts, excluded counts, per-class counts, and exact Train / Val / Test split sizes per class.

### Files to Upload to Colab (or mount from Google Drive)
1. **`rsna_positives.zip`** — Zip of real RSNA training PNGs (e.g. the RSNA `boneage-training-dataset` folder, or 5,500+ PNGs from it).
2. **`validator_negatives.zip`** — Zip of real negative images (non-hand X-rays, natural photos, animals, documents, screenshots).
3. *(Optional)* **`should_reject.zip`** or **`validator_excluded_eval_files.json`** — If you placed negative test files in `tests/validator/should_reject`, upload `should_reject.zip` (or list their filenames in `SHOULD_REJECT_FILES` below) so they are guaranteed excluded from training negatives.

### Output Produced
- `/content/hand_validator_best.pt` — Download and place at `C:\bonetest\backend\checkpoints\hand_validator_best.pt`.

In [ ]:
# 1. Check GPU & Unpack Uploaded Zips
import hashlib, json, os, zipfile
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

POS_DIR = Path('/content/data/positives')
NEG_DIR = Path('/content/data/negatives')
SHOULD_ACCEPT_DIR = Path('/content/data/should_accept')
SHOULD_REJECT_DIR = Path('/content/data/should_reject')
for d in (POS_DIR, NEG_DIR, SHOULD_ACCEPT_DIR, SHOULD_REJECT_DIR):
    d.mkdir(parents=True, exist_ok=True)

for zip_name, target_dir in [
    ('/content/rsna_positives.zip', POS_DIR),
    ('/content/validator_negatives.zip', NEG_DIR),
    ('/content/should_accept.zip', SHOULD_ACCEPT_DIR),
    ('/content/should_reject.zip', SHOULD_REJECT_DIR),
]:
    zp = Path(zip_name)
    if zp.exists():
        print(f'Extracting {zp} -> {target_dir} ...')
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(target_dir)


In [ ]:
# 2. Shared Model Architecture & Preprocessing (Exact Match with backend/app/models/validator.py)
from typing import List
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

VALIDATOR_CLASSES: List[str] = [
    'valid_pediatric_hand_xray',
    'other',
]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def letterbox_rgb_224(image_rgb: np.ndarray, target_size: int = 224) -> np.ndarray:
    h, w = image_rgb.shape[:2]
    scale = target_size / float(max(h, w))
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2
    canvas[y_off : y_off + new_h, x_off : x_off + new_w] = resized
    return canvas

def preprocess_validator_image(image_bgr: np.ndarray, target_size: int = 224) -> torch.Tensor:
    if image_bgr.ndim == 2:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_GRAY2RGB)
    elif image_bgr.shape[2] == 4:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGRA2RGB)
    else:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    letterboxed = letterbox_rgb_224(rgb, target_size=target_size)
    norm = letterboxed.astype(np.float32) / 255.0
    norm = (norm - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(norm.transpose(2, 0, 1)).float()

class HandXRayValidator(nn.Module):
    def __init__(self, num_classes: int = len(VALIDATOR_CLASSES), dropout_rate: float = 0.3, pretrained: bool = False) -> None:
        super().__init__()
        self.num_classes = num_classes
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, num_classes),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.extract_features(x))

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.forward(x), dim=-1)


In [ ]:
# 3. Exclusion Lists (30 should_accept + 200 Stage 1/3 check images + should_reject negatives)
MAX_POSITIVES = 5000

# Exact 30 files in tests/validator/should_accept
SHOULD_ACCEPT_FILES = {
    '11621.png', '11768.png', '13224.png', '13428.png', '13991.png', '2098.png',
    '2482.png', '2592.png', '2766.png', '2839.png', '2996.png', '3096.png',
    '3133.png', '3477.png', '3579.png', '3623.png', '3695.png', '3713.png',
    '3858.png', '3969.png', '4244.png', '4260.png', '5371.png', '5445.png',
    '5519.png', '5711.png', '5778.png', '7960.png', '8748.png', '8803.png'
}

# Exact 200 random check images from scripts/check_200_rsna_stage1_stage3.py (seed=101)
CHECK_200_FILES = {
    '1434.png', '1627.png', '1748.png', '1779.png', '1797.png', '1806.png', '1909.png', '1947.png', '1967.png', '2058.png',
    '2111.png', '2112.png', '2116.png', '2197.png', '2348.png', '2364.png', '2421.png', '2450.png', '2490.png', '2551.png',
    '2568.png', '2644.png', '2718.png', '3007.png', '3022.png', '3122.png', '3167.png', '3178.png', '3190.png', '3192.png',
    '3445.png', '3520.png', '3531.png', '3566.png', '3665.png', '3693.png', '3804.png', '3887.png', '3937.png', '3962.png',
    '3978.png', '4186.png', '4215.png', '4259.png', '4280.png', '4315.png', '4333.png', '4346.png', '4611.png', '4618.png',
    '4676.png', '4858.png', '5103.png', '5191.png', '5315.png', '5364.png', '5423.png', '5567.png', '5681.png', '5752.png',
    '5766.png', '5772.png', '5809.png', '5841.png', '5986.png', '6006.png', '6029.png', '6037.png', '6071.png', '6293.png',
    '6378.png', '6383.png', '6495.png', '6591.png', '6816.png', '6983.png', '7041.png', '7124.png', '7178.png', '7396.png',
    '7426.png', '7547.png', '7626.png', '7630.png', '7642.png', '7901.png', '7903.png', '7918.png', '7966.png', '8051.png',
    '8095.png', '8105.png', '8127.png', '8130.png', '8132.png', '8198.png', '8240.png', '8327.png', '8345.png', '8385.png',
    '8388.png', '8407.png', '8464.png', '8489.png', '8506.png', '8532.png', '8567.png', '8602.png', '8628.png', '8814.png',
    '8858.png', '8879.png', '9277.png', '9456.png', '9569.png', '9581.png', '9925.png', '10108.png', '10117.png', '10176.png',
    '10226.png', '10261.png', '10339.png', '10403.png', '10438.png', '10592.png', '10641.png', '10820.png', '10889.png', '10996.png',
    '11054.png', '11094.png', '11155.png', '11207.png', '11210.png', '11216.png', '11240.png', '11295.png', '11417.png', '11500.png',
    '11558.png', '11645.png', '11676.png', '11750.png', '11769.png', '12044.png', '12069.png', '12198.png', '12226.png', '12244.png',
    '12467.png', '12567.png', '12581.png', '12683.png', '12722.png', '12755.png', '12768.png', '12819.png', '12831.png', '12962.png',
    '13055.png', '13228.png', '13283.png', '13325.png', '13331.png', '13526.png', '13543.png', '13556.png', '13586.png', '13605.png',
    '13697.png', '13726.png', '13764.png', '13827.png', '13839.png', '13887.png', '13897.png', '13932.png', '13961.png', '13970.png',
    '14032.png', '14051.png', '14173.png', '14333.png', '14429.png', '14520.png', '14608.png', '14724.png', '14815.png', '14910.png',
    '14983.png', '14997.png', '15021.png', '15029.png', '15155.png', '15160.png', '15417.png', '15534.png', '15545.png', '15604.png'
}

SHOULD_REJECT_FILES = set()

# Also merge from validator_excluded_eval_files.json or uploaded should_accept / should_reject folders if present
manifest_json = Path('/content/validator_excluded_eval_files.json')
if manifest_json.exists():
    mdata = json.loads(manifest_json.read_text())
    SHOULD_ACCEPT_FILES.update(mdata.get('should_accept_files', []))
    CHECK_200_FILES.update(mdata.get('check_200_files', []))
    SHOULD_REJECT_FILES.update(mdata.get('should_reject_files', []))

if SHOULD_ACCEPT_DIR.exists():
    SHOULD_ACCEPT_FILES.update(p.name for p in SHOULD_ACCEPT_DIR.rglob('*') if p.is_file())

SHOULD_REJECT_HASHES = set()
if SHOULD_REJECT_DIR.exists():
    for p in SHOULD_REJECT_DIR.rglob('*'):
        if p.is_file():
            SHOULD_REJECT_FILES.add(p.name)
            SHOULD_REJECT_HASHES.add(hashlib.sha256(p.read_bytes()).hexdigest())

EXCLUDED_POS_FILES = SHOULD_ACCEPT_FILES | CHECK_200_FILES
print(f'Configured excluded RSNA positive filenames : {len(EXCLUDED_POS_FILES)} (should_accept={len(SHOULD_ACCEPT_FILES)}, check_200={len(CHECK_200_FILES)})')
print(f'Configured excluded negative filenames      : {len(SHOULD_REJECT_FILES)} (hashes={len(SHOULD_REJECT_HASHES)})')


In [ ]:
# 4. Filter Real Datasets, Sample up to 5000 Positives, Log Counts & Train on GPU
import random
from torch.utils.data import Dataset, DataLoader

EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}
raw_pos_files = sorted([p for p in POS_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTS])
raw_neg_files = sorted([p for p in NEG_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXTS])

# Exclude should_accept (30) + check_200 (200) from positives
pos_files = [p for p in raw_pos_files if p.name not in EXCLUDED_POS_FILES]
excluded_pos_count = len(raw_pos_files) - len(pos_files)

# Exclude should_reject files (by filename and SHA-256 hash) from negatives
neg_files = []
excluded_neg_count = 0
for p in raw_neg_files:
    if p.name in SHOULD_REJECT_FILES:
        excluded_neg_count += 1
        continue
    if SHOULD_REJECT_HASHES and hashlib.sha256(p.read_bytes()).hexdigest() in SHOULD_REJECT_HASHES:
        excluded_neg_count += 1
        continue
    neg_files.append(p)

rng = random.Random(42)
rng.shuffle(pos_files)
rng.shuffle(neg_files)

# Default to 5000 RSNA positives
if MAX_POSITIVES and len(pos_files) > MAX_POSITIVES:
    pos_files = pos_files[:MAX_POSITIVES]

print('=' * 85)
print('DATASET FILTERING & CLASS COUNT SUMMARY')
print('=' * 85)
print(f'Raw positive RSNA files found       : {len(raw_pos_files)}')
print(f'Excluded held-out positive files    : {excluded_pos_count} (from should_accept + check_200)')
print(f'Selected positive RSNA files (cap)  : {len(pos_files)} (MAX_POSITIVES={MAX_POSITIVES})')
print(f'Raw negative files found            : {len(raw_neg_files)}')
print(f'Excluded should_reject negatives    : {excluded_neg_count}')
print(f'Selected negative files             : {len(neg_files)}')
assert len(pos_files) > 0 and len(neg_files) > 0, 'Both real positive and real negative images are required!'

def split_85_15(items):
    n = len(items)
    n_tr = int(n * 0.85)
    return items[:n_tr], items[n_tr:]

p_tr, p_val = split_85_15([(p, 0) for p in pos_files])
n_tr, n_val = split_85_15([(p, 1) for p in neg_files])
train_items = p_tr + n_tr
val_items = p_val + n_val
rng.shuffle(train_items)

print('-' * 85)
print('TRAIN / VAL SPLIT SIZES PER CLASS:')
print(f'  Class 0 (valid_pediatric_hand_xray) : Total={len(pos_files):5d} | Train={len(p_tr):5d} | Val={len(p_val):5d}')
print(f'  Class 1 (other)                     : Total={len(neg_files):5d} | Train={len(n_tr):5d} | Val={len(n_val):5d}')
print(f'  Combined Total                      : Total={len(pos_files)+len(neg_files):5d} | Train={len(train_items):5d} | Val={len(val_items):5d}')
print('=' * 85)

class DiskDataset(Dataset):
    def __init__(self, items, augment=False):
        self.items = items
        self.augment = augment
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if self.augment and random.random() < 0.5:
            img = cv2.flip(img, 1)
        return preprocess_validator_image(img, 224), torch.tensor(label, dtype=torch.long)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(DiskDataset(train_items, augment=True), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(DiskDataset(val_items, augment=False), batch_size=32, shuffle=False, num_workers=2)

model = HandXRayValidator(num_classes=2, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

best_val_loss = float('inf')
for epoch in range(1, 11):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    vl_loss, vl_corr, vl_tot = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            vl_loss += criterion(out, y).item() * x.size(0)
            vl_corr += (out.argmax(1) == y).sum().item()
            vl_tot += x.size(0)
    vl_loss /= max(1, vl_tot)
    vl_acc = vl_corr / max(1, vl_tot)
    print(f'Epoch {epoch:02d}/10 | Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc*100:.2f}%')
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'classes': list(VALIDATOR_CLASSES),
            'val_loss': vl_loss,
            'val_acc': vl_acc,
            'epoch': epoch,
            'train_counts': {'valid_pediatric_hand_xray': len(p_tr), 'other': len(n_tr)},
            'val_counts': {'valid_pediatric_hand_xray': len(p_val), 'other': len(n_val)},
            'excluded_positives_count': len(EXCLUDED_POS_FILES),
            'excluded_negatives_count': excluded_neg_count,
            'model_architecture': 'ResNet18-2Class-Validator',
        }, '/content/hand_validator_best.pt')


In [ ]:
# 5. Held-Out Validation ROC, RSNA Accept Rate, False-Accept Rate
from sklearn.metrics import roc_auc_score

ckpt = torch.load('/content/hand_validator_best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'], strict=True)
model.eval()

y_true_pos, p_hand = [], []
with torch.no_grad():
    for x, y in val_loader:
        probs = model.predict_proba(x.to(device)).cpu().numpy()
        p_hand.extend(probs[:, 0].tolist())
        y_true_pos.extend((y.numpy() == 0).astype(int).tolist())

y_true_pos = np.array(y_true_pos)
p_hand = np.array(p_hand)
auc = roc_auc_score(y_true_pos, p_hand)
print(f'Validation ROC AUC: {auc:.4f}')
for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    accept_pos = np.mean(p_hand[y_true_pos == 1] >= thr) * 100
    false_accept_neg = np.mean(p_hand[y_true_pos == 0] >= thr) * 100
    print(f'Threshold={thr:.2f} | RSNA Accept Rate={accept_pos:6.2f}% | False-Accept Rate={false_accept_neg:6.2f}%')
